# RAG básico passo à passo

## 1) Carregamento de bibliotecas

In [1]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnableParallel
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from IPython import display
import os
import httpx
import certifi
import ssl

C:\Users\francisco.bneto\AppData\Local\Temp\ipykernel_3052\2181131115.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, TextLoader


In [2]:
http_client = httpx.Client(verify=False)

In [3]:
os.chdir(r'c:\Users\francisco.bneto\Documents\gen-ai-formation')
print(os.getcwd())

c:\Users\francisco.bneto\Documents\gen-ai-formation


In [4]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
print("Chave carregada:", api_key[:10] + "...")

Chave carregada: sk-or-v1-c...


In [5]:
# modelo de LLM
llm_model = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1",
    http_client=http_client
)

# modelo de embeddings
embedding_model = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    openai_api_key=api_key,
    openai_api_base="https://openrouter.ai/api/v1",
    http_client=http_client
)

## 2) Baixar documentos e vetorizar embeddings

In [6]:
doc = TextLoader('./documentos/GTB_gold_Nov23.txt', encoding='utf-8').load()

doc[0].page_content[:400]

'\n1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para port'

In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = splitter.split_documents(doc)

In [8]:
chunks[2]

Document(metadata={'source': './documentos/GTB_gold_Nov23.txt'}, page_content='autarquia, incentivo ou recomendação de sua comercialização. \n \nA cobertura dos seguros/benefícios ou serviços aqui descritos serão anulados, seja antes ou depois que \numa perda ou pedido de serviços forem efetuados, se você intencionalmente ocultar ou fizer \ninterpretação  errônea  de  qualquer  fato  material  ou  circunstância,  ou  fornecer  informação  fraudulenta \nrelativa  aos  planos  de  seguro  ou  outros  serviços  aqui  descritos  para:  A  Mastercard  International,  a \nEmpresa de Seguros, a instituição financeira que emitiu a Conta do cartão ou qualquer outra empresa \nque estiver prestando serviços e/ou administração em nome destes programas. \n \nPara  dar  entrada  em  uma  ocorrência/sinistro  ou  para  obter  mais  informações  sobre  qualquer um \ndesses serviços, ligue para o número gratuito do Mastercard Global Service™ específico para o seu país, \nou ligue a cobrar para os Estad

In [9]:
vectorstore = InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [20]:
retriever = vectorstore.as_retriever(
    search_kwargs={'k': 4}
)

In [21]:
retriever.invoke("Seguro viagem")

[Document(id='16b2df4b-2aa1-4fb9-8abe-9d00dd5baf6d', metadata={'source': './documentos/GTB_gold_Nov23.txt'}, page_content='2 \nVersão: novembro 2023 \n2021 \n \n \n \nGuia de Benefícios Mastercard \nBenefícios que estão sempre com você. \n \n \n \nÍndice \n \n \nContents \nPrograma de Cartão da Edição Mastercard Gold ...................................................................... 1 \nGuia de Benefícios Mastercard ............................................................................................ 2 \nÍndice .................................................................................................................................... 2 \nSeguro Garantia Estendida Original .............................................................................. 3 \nCompra Protegida ........................................................................................................ 8 \nProteção de Preço .........................................................................

## 3) Receber consulta do usuário e fazer Embedding

In [12]:
query = "Como eu devo proceder caso tenha um item roubado?"

query_embed = embedding_model.embed_query(query)
query_embed

[0.0189208984375,
 0.04254150390625,
 -0.0180206298828125,
 0.041900634765625,
 -0.003452301025390625,
 0.006855010986328125,
 -0.003894805908203125,
 -0.01117706298828125,
 -0.0204315185546875,
 -0.0212554931640625,
 0.057891845703125,
 0.0084075927734375,
 0.0201568603515625,
 -0.043304443359375,
 0.02520751953125,
 0.0033416748046875,
 -0.00463104248046875,
 -0.017822265625,
 -0.0234527587890625,
 0.01494598388671875,
 -0.03204345703125,
 0.044921875,
 0.01021575927734375,
 -0.0019369125366210938,
 -0.01064300537109375,
 -0.0423583984375,
 -0.0067901611328125,
 -0.0040130615234375,
 0.018524169921875,
 0.0010290145874023438,
 0.0523681640625,
 -0.0177459716796875,
 -0.0167999267578125,
 0.0012845993041992188,
 0.02130126953125,
 0.039947509765625,
 -0.017242431640625,
 -0.034088134765625,
 0.03656005859375,
 0.0193634033203125,
 0.0160369873046875,
 0.0219879150390625,
 0.043548583984375,
 -0.0112152099609375,
 0.0168914794921875,
 0.012054443359375,
 0.005977630615234375,
 -0.02017

## 4) Buscar embeddings da consulta no banco vetorial

In [13]:
similar_chunks = retriever.invoke(query)
similar_chunks

[Document(id='1062fd06-e7e2-48b0-b5f1-2b6416084e2d', metadata={'source': './documentos/GTB_gold_Nov23.txt'}, page_content='autarquia, incentivo ou recomendação de sua comercialização. \n \nA cobertura dos seguros/benefícios ou serviços aqui descritos serão anulados, seja antes ou depois que \numa perda ou pedido de serviços forem efetuados, se você intencionalmente ocultar ou fizer \ninterpretação  errônea  de  qualquer  fato  material  ou  circunstância,  ou  fornecer  informação  fraudulenta \nrelativa  aos  planos  de  seguro  ou  outros  serviços  aqui  descritos  para:  A  Mastercard  International,  a \nEmpresa de Seguros, a instituição financeira que emitiu a Conta do cartão ou qualquer outra empresa \nque estiver prestando serviços e/ou administração em nome destes programas. \n \nPara  dar  entrada  em  uma  ocorrência/sinistro  ou  para  obter  mais  informações  sobre  qualquer um \ndesses serviços, ligue para o número gratuito do Mastercard Global Service™ específico para o

## 5) Resgatar os documentos similares do banco

In [14]:
similar_text = [
    chunk.page_content for chunk in similar_chunks
]

similar_text

['autarquia, incentivo ou recomendação de sua comercialização. \n \nA cobertura dos seguros/benefícios ou serviços aqui descritos serão anulados, seja antes ou depois que \numa perda ou pedido de serviços forem efetuados, se você intencionalmente ocultar ou fizer \ninterpretação  errônea  de  qualquer  fato  material  ou  circunstância,  ou  fornecer  informação  fraudulenta \nrelativa  aos  planos  de  seguro  ou  outros  serviços  aqui  descritos  para:  A  Mastercard  International,  a \nEmpresa de Seguros, a instituição financeira que emitiu a Conta do cartão ou qualquer outra empresa \nque estiver prestando serviços e/ou administração em nome destes programas. \n \nPara  dar  entrada  em  uma  ocorrência/sinistro  ou  para  obter  mais  informações  sobre  qualquer um \ndesses serviços, ligue para o número gratuito do Mastercard Global Service™ específico para o seu país, \nou ligue a cobrar para os Estados Unidos no número 1-636-722-8881 (Português).',
 'Proteção de Preço .......

## 6) Aumentar o prompt com consulta e documentos

In [15]:
prompt_template = ChatPromptTemplate(
    [("system", "Responda usando exclusivamente os conteúdos fornecidos. \n\nContexto:\n{contexto}"),
    ("human", "{query}")
    ]
)

## 7) Gerar a Resposta

In [16]:
response = llm_model.invoke(query)

print(response.content)

Se você tiver um item roubado, é importante agir rapidamente para aumentar suas chances de recuperá-lo e evitar problemas legais. Aqui está um passo a passo para você seguir:

1. **Verifique o seu seguro**: Se você tiver seguro de responsabilidade civil, que inclui a proteção contra roubo, você pode ter direito a uma indenização por valor do item roubado.

2. **Reporte o roubo**: Informe a polícia sobre o roubo. Isso é crucial para iniciar a investigação e aumentar suas chances de recuperar o item. A polícia pode ter acesso a câmeras de segurança, testemunhas e outros detalhes que podem ajudar a identificar o ladrão.

3. **Peça a assistência da polícia**: A polícia pode oferecer assistência em várias formas, como fornecer informações sobre o paradeiro do item, ajudar a identificar o ladrão ou fornecer orientação sobre como proceder.

4. **Contate as autoridades locais**: Dependendo do local onde o roubo ocorreu, você pode precisar entrar em contato com as autoridades locais, como o Dep

## 8) Criação da Chain

In [17]:
chain = prompt_template | llm_model | StrOutputParser()

In [22]:
chunks = retriever.invoke(query)

contexto = "\n\n".join(chunk.page_content for chunk in chunks)

chain.invoke({
    "query": query,
    "contexto": contexto
})

'Se você tiver um item roubado, você deve proceder da seguinte maneira:\n\n1. Ligue para o número gratuito do Mastercard Global Service™ específico para o seu país, ou ligue a cobrar para os Estados Unidos no número 1-636-722-8881 (Português).\n\nEssa é a primeira etapa para obter mais informações e proceder com a solicitação de sinistro.'